#**Data continue**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
from google.colab import files

# Preprocesamiento y Balanceo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from imblearn.over_sampling import ADASYN

# Modelos
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import GradientBoostingClassifier

# Métricas
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.inspection import permutation_importance

# --- CARGA DE DATOS ---
print("Por favor, sube el archivo 'telecom_x_limpio.csv' que descargaste en la Parte 1:")
uploaded = files.upload()

# Leemos el archivo subido usando io.BytesIO
df = pd.read_csv(io.BytesIO(uploaded['telecom_x_limpio.csv']))
print("\n¡Datos cargados correctamente! Dimensiones:", df.shape)

Limpieza y encoding

In [ ]:
# 1. Eliminar columnas irrelevantes
df = df.drop(columns=['ID_Cliente', 'Churn'], errors='ignore')

# 2. Separar variables explicativas (X) de la variable objetivo (y)
X = df.drop('Churn_Binario', axis=1)
y = df['Churn_Binario']

# 3. Identificar columnas categóricas y numéricas
cols_cat = X.select_dtypes(include='object').columns.tolist()
cols_num = X.select_dtypes(exclude='object').columns.tolist()

# 4. ColumnTransformer: OneHotEncoder para categóricas, passthrough para numéricas
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cols_cat)
], remainder='passthrough')

X_encoded_arr = preprocessor.fit_transform(X)

# Reconstruir nombres de columnas
ohe_cols = preprocessor.named_transformers_['cat'].get_feature_names_out(cols_cat).tolist()
all_cols  = ohe_cols + cols_num
X_encoded = pd.DataFrame(X_encoded_arr, columns=all_cols, index=X.index)

print(f"Dimensiones de X procesado (con dummies): {X_encoded.shape}")
print(f"Proporción de Churn (Cancelación) actual:\n{y.value_counts(normalize=True)*100}")

Análisis de Correlación y Gráficos Dirigidos

In [ ]:
from sklearn.feature_selection import mutual_info_classif

# Información mutua entre cada variable y el objetivo
mask = y.notna()
mi_scores = mutual_info_classif(X_encoded[mask].fillna(0), y[mask], random_state=42)
mi_series = pd.Series(mi_scores, index=X_encoded.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
top_mi = mi_series.head(11)
sns.barplot(x=top_mi.values, y=top_mi.index, palette='coolwarm')
plt.title('Relevancia de Variables con la Cancelación (Churn) — Información Mutua')
plt.xlabel('Información Mutua')
plt.tight_layout()
plt.show()

# Gráficos dirigidos (Violin plots)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.violinplot(x='Churn_Binario', y='Meses_Contrato', data=df, ax=axes[0], palette='pastel', inner='quartile')
axes[0].set_title('Meses de Contrato vs Cancelación')

sns.violinplot(x='Churn_Binario', y='Cobro_Total', data=df, ax=axes[1], palette='pastel', inner='quartile')
axes[1].set_title('Cobro Total vs Cancelación')
plt.show()

Separación, Balanceo (ADASYN) y Normalización

In [ ]:
# --------------------------------------------------------
# PARCHE DE SEGURIDAD: Eliminar valores nulos (NaN)
# --------------------------------------------------------
# 1. Conservamos solo las filas donde 'y' NO es nulo
mascara_no_nulos = y.notna()
X_encoded = X_encoded[mascara_no_nulos]
y = y[mascara_no_nulos]

# 2. Por seguridad, rellenamos cualquier nulo accidental en X con 0
X_encoded = X_encoded.fillna(0)

print(f"Borrados los valores nulos. Nuevas dimensiones de X: {X_encoded.shape}")
print(f"Nuevas dimensiones de y: {y.shape}")
print("-" * 50)

# --------------------------------------------------------
# AHORA SÍ: Separación, Balanceo (ADASYN) y Normalización
# --------------------------------------------------------
# 1. División Train/Test (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.30, random_state=42, stratify=y
)

# 2. Balanceo con ADASYN (Aplicado SOLO al conjunto de entrenamiento)
adasyn = ADASYN(random_state=42)
X_train_bal, y_train_bal = adasyn.fit_resample(X_train, y_train)
X_train_bal = pd.DataFrame(X_train_bal, columns=X_encoded.columns)
y_train_bal = pd.Series(y_train_bal)

# 3. Normalización / Escalado Min-Max
scaler = MinMaxScaler()

# Lista de tus columnas estrictamente numéricas (las que no son 0 y 1)
cols_numericas = ['Meses_Contrato', 'Cobro_Mensual', 'Cobro_Total', 'Cuentas_Diarias', 'Cant_Servicios']

# Ajustamos el escalador con el entrenamiento y transformamos ambos
X_train_bal[cols_numericas] = scaler.fit_transform(X_train_bal[cols_numericas])
X_test[cols_numericas] = scaler.transform(X_test[cols_numericas])

print(f"Distribución de Churn en Entrenamiento después de ADASYN:\n{y_train_bal.value_counts()}")

Entrenamiento y Evaluación de Modelos

In [ ]:
# --- ENTRENAMIENTO ---
# Modelo 1: SGD Classifier (equivalente lineal a Regresión Logística)
modelo_sgd = SGDClassifier(loss='log_loss', random_state=42, max_iter=1000, tol=1e-3)
modelo_sgd.fit(X_train_bal, y_train_bal)
y_pred_sgd = modelo_sgd.predict(X_test)

# Modelo 2: Gradient Boosting
modelo_gb = GradientBoostingClassifier(random_state=42, n_estimators=100, max_depth=4, learning_rate=0.1)
modelo_gb.fit(X_train_bal, y_train_bal)
y_pred_gb = modelo_gb.predict(X_test)

# --- EVALUACIÓN (Función auxiliar) ---
def reporte_modelo(nombre, y_verdadero, y_predicho):
    print(f"\n{'='*50}")
    print(f"REPORTE DE RENDIMIENTO: {nombre}")
    print(f"{'='*50}")
    print(classification_report(y_verdadero, y_predicho))

    plt.figure(figsize=(4,3))
    sns.heatmap(confusion_matrix(y_verdadero, y_predicho), annot=True, fmt='d', cmap='Blues')
    plt.title(f'Matriz de Confusión - {nombre}')
    plt.ylabel('Real')
    plt.xlabel('Predicho')
    plt.show()

reporte_modelo("SGD CLASSIFIER (LOG LOSS)", y_test, y_pred_sgd)
reporte_modelo("GRADIENT BOOSTING", y_test, y_pred_gb)

Interpretación Estratégica

In [ ]:
# Importancia de Variables en Gradient Boosting via Permutation Importance
result = permutation_importance(modelo_gb, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
importancias = pd.DataFrame({
    'Variable': X_train_bal.columns,
    'Importancia': result.importances_mean
}).sort_values(by='Importancia', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importancia', y='Variable', data=importancias.head(10), palette='viridis')
plt.title('Top 10 Variables más Importantes (Gradient Boosting — Permutation Importance)')
plt.show()

# Coeficientes del SGD Classifier
coeficientes = pd.DataFrame({
    'Variable': X_train_bal.columns,
    'Coeficiente': modelo_sgd.coef_[0]
}).sort_values(by='Coeficiente', ascending=False)

print("\n--- INSIGHTS ESTRATÉGICOS (SGD CLASSIFIER) ---")
print("\nFactores que MÁS AUMENTAN la probabilidad de cancelar:")
print(coeficientes.head(5))

print("\nFactores que MÁS AYUDAN A RETENER al cliente:")
print(coeficientes.tail(5))